# AdPilot Pro – A/B Testing & Multi-Armed Bandits Simulation Notebook

This notebook details the statistical lift, Bayesian posterior parameter estimations, and Thompson Sampling bandit policy comparisons.


## 1. Business Case & Objectives



### A/B Testing & Bandit Frameworks:
Evaluate corporate marketing creative performance through real-time traffic allocation. Transitioning from classical A/B testing (static split) to dynamic reinforcement learning-based Multi-Armed Bandits (Thompson Sampling, UCB) to minimize regret.



## 2. Ingestion & Real Data Loading


In [ ]:
import numpy as np
import pandas as pd
import os

# Load clean campaign conversions from the workspace
clean_path = "research/datasets/processed/analytics_clean.csv"
if os.path.exists(clean_path):
    df_clean = pd.read_csv(clean_path)
    # Segment by communication channel (e.g., cellular vs telephone) to run real A/B split comparison
    clicks_a = (df_clean[df_clean["contact"] == "cellular"]["y"] == "yes").astype(int).dropna().tolist()[:1000]
    clicks_b = (df_clean[df_clean["contact"] == "telephone"]["y"] == "yes").astype(int).dropna().tolist()[:1000]
else:
    print("Warning: analytics_clean.csv not found. Falling back to synthetic clicks.")
    clicks_a = np.random.binomial(1, 0.05, 1000).tolist()
    clicks_b = np.random.binomial(1, 0.062, 1000).tolist()

df = pd.DataFrame({
    "group_a": clicks_a[:min(len(clicks_a), len(clicks_b))],
    "group_b": clicks_b[:min(len(clicks_a), len(clicks_b))]
})

print(f"Real Campaign Success Rate Group A (Cellular): {np.mean(clicks_a):.4f}")
print(f"Real Campaign Success Rate Group B (Telephone): {np.mean(clicks_b):.4f}")


## 3. Classical A/B Testing


In [ ]:
from scipy import stats

# Perform two-sample t-test to check statistical significance
t_stat, p_val = stats.ttest_ind(clicks_a, clicks_b)
lift = (np.mean(clicks_b) - np.mean(clicks_a)) / (np.mean(clicks_a) + 1e-9)

print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {p_val:.4f}")
print(f"Relative Lift: {lift * 100.0:.2f}%")


## 4. Thompson Sampling Bandit Policy


In [ ]:
# Implement Multi-Armed Bandit using Thompson Sampling (Beta-Bernoulli conjugate priors)
n_trials = 500
n_arms = 2
alpha = np.ones(n_arms)
beta = np.ones(n_arms)

true_rates = [np.mean(clicks_a), np.mean(clicks_b)]
regrets = []
selections = []

for t in range(n_trials):
    # Sample from posteriors
    theta = [np.random.beta(alpha[i], beta[i]) for i in range(n_arms)]
    chosen_arm = np.argmax(theta)
    selections.append(chosen_arm)
    
    # Observe reward
    reward = np.random.binomial(1, true_rates[chosen_arm])
    
    # Update priors
    if reward == 1:
        alpha[chosen_arm] += 1
    else:
        beta[chosen_arm] += 1
        
    # Calculate regret
    regrets.append(max(true_rates) - true_rates[chosen_arm])

cum_regret = np.cumsum(regrets)
print(f"Thompson Sampling cumulative regret at t={n_trials}: {cum_regret[-1]:.4f}")
print(f"Arm selections distribution: Arm 0: {selections.count(0)}, Arm 1: {selections.count(1)}")


## 5. Export Model Checkpoint & Reports


In [ ]:
import joblib
import json
import os
from datetime import datetime

out_dir = "research/models/ab_testing"
os.makedirs(out_dir, exist_ok=True)

# Export bandit status
bandit_checkpoint = {
    "policy": "Thompson_Sampling",
    "alpha": alpha.tolist(),
    "beta": beta.tolist(),
    "n_arms": n_arms
}
joblib.dump(bandit_checkpoint, os.path.join(out_dir, "bandit_model.pkl"))

# Export feature schema
schema = {
    "input_types": {
        "alpha": "list[float]",
        "beta": "list[float]"
    },
    "actions": "arm_index"
}
with open(os.path.join(out_dir, "feature_schema.json"), "w") as f:
    json.dump(schema, f, indent=2)

# Export metadata
metadata = {
    "experiment_name": "Multi_Armed_Bandits_Thompson_Sampling",
    "version": "1.0.0",
    "n_trials": n_trials,
    "training_date": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
}
with open(os.path.join(out_dir, "metadata.json"), "w") as f:
    json.dump(metadata, f, indent=2)

# Export metrics
metrics = {
    "cumulative_regret": float(cum_regret[-1]),
    "p_value": float(p_val),
    "lift": float(lift)
}
with open(os.path.join(out_dir, "metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)

print("All requested A/B Testing & Bandit assets successfully exported to research/models/ab_testing/.")
